# Clinical guideline assistant (v3.1)
Run the cells in order. Add `MONGODB_URI` and `ACCESS_TOKEN` in Colab **Secrets** (key icon). Runtime: GPU (L4 or better).

In [3]:
!pip -q install fastapi uvicorn nest_asyncio transformers accelerate pymongo bitsandbytes sentence-transformers httpx pytest
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [1]:
from google.colab import files
name = next(iter(files.upload()))   # choose the app zip (clinical-rag-v3.1.zip)
!rm -rf clinical-rag && mkdir clinical-rag && unzip -qo "{name}" -d clinical-rag
!pip -q install -r clinical-rag/requirements.txt

Saving clinical-rag-v3.1.zip to clinical-rag-v3.1.zip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.41.3 which is incompatible.
python-fasthtml 0.14.16 requires starlette>=1.0.1, but you have starlette 0.41.3 which is incompatible.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you have fastapi 0.115.6 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.41.3 which is incompatible.


In [4]:
# 1) Model server on 127.0.0.1:8000 (local only, never tunnelled). Takes a few minutes the first time.
%run clinical-rag/colab_server.py

⏳ Loading PubMedBERT and BGE reranker on cuda...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

... (29 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model server running on 127.0.0.1:8000 (local only)


In [5]:
# 2) Offline tests of the deterministic parts and the verifier
!cd clinical-rag && python -m pytest -q tests

...............                                                          [100%]
15 passed in 0.31s


In [6]:
# 3) App + public tunnel for the chat page only
import os, re, subprocess
from google.colab import userdata
os.environ['MONGODB_URI'] = userdata.get('MONGODB_URI')
os.environ['ACCESS_TOKEN'] = userdata.get('ACCESS_TOKEN')
os.environ['COLAB_RETRIEVE_URL'] = 'http://127.0.0.1:8000'
os.environ['COLAB_GENERATE_URL'] = 'http://127.0.0.1:8000/generate'
os.environ['COLAB_TIMEOUT'] = '300'
subprocess.Popen(['uvicorn', 'app:app', '--port', '7860'], cwd='clinical-rag')
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:7860'], stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: print('Open:', m.group(0)); break

Open: https://contamination-alignment-settled-puts.trycloudflare.com


In [8]:
# 4) Accuracy scorecard on the real backend. --show prints every verified and rejected claim.
!cd clinical-rag && python tests/run_eval.py --real --show

Traceback (most recent call last):
  File "/content/clinical-rag/tests/run_eval.py", line 93, in <module>
    ok = all([asyncio.run(run_case(p, a.real, a.show)) for p in paths])
              ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/base_events.py", line 726, in run_until_complete
    return future.result()
           ~~~~~~~~~~~~~^^
  File "/content/clinical-rag/tests/run_eval.py", line 67, in run_case
    deterministic(case, rows)
    ~~~~~~~~~~~~~^^^^^^^^^^^^
  File "/content/clinical-rag/tests/run_eval.py", line 22, in deterministic
    ex, fx = case["expect"], facts.analyze_facts(case["input"])
             ~~~~^^^^^^^^^^
TypeError: list indices must be integers or sli

In [ ]:
# 5) Download the full results (JSON) to review the claims or send them for analysis
!cd clinical-rag/tests && zip -qr ../eval_results.zip out
from google.colab import files
files.download('clinical-rag/eval_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>